In [1]:
# Celda 1
%cd C:/Users/fabi_/Documents/workflow/textos-catolicos-xix

import os
import pickle
import random
import re
import sys
import tempfile
from collections import Counter
import pandas as pd
import pymupdf  # Turns a PDF page into an image
from spylls.hunspell import Dictionary  # Spanish Hunspell dictionary in pure Python

sys.path.insert(0, "ocr-piloto")
import reglas_modernizacion_postcorreccion as reglas

CLEAN_FOLDER = "textos-limpios"                                    # From 5-limpieza
PDF_FOLDER = "E:/Datos/textos-catolicos-xix/biblioteca_religiosa"  # Only for the page images
DICTIONARY = "ocr-piloto/diccionario-es/es_ES"
OUT_FOLDER = "ocr-piloto/paginas-baja-calidad"                     # Page images, clean text and modernized text
MIN_BOOKS = 10      # A word in 10 books or more of the corpus exists, even if the modern dictionary lacks it ("dixo")
WORST_BOOKS = 10    # One random page from each of the 10 worst books
MIN_WORDS = 150     # Pages with fewer words (title pages, plates, chapter ends) are not drawn
SEED = 2026

C:\Users\fabi_\Documents\workflow\textos-catolicos-xix


In [2]:
# Celda 2
# Words of each book, without tags, page breaks or digits.:
WORD = re.compile(r"[^\W\d_]+")

def book_words(path):
    with open(path, encoding="utf-8") as f:
        text = re.sub(r"<[^>]+>", " ", f.read())
    return [word.lower() for word in WORD.findall(text)]

words_by_book = {}
for file_name in sorted(os.listdir(CLEAN_FOLDER)):
    if file_name.endswith(".txt"):
        words_by_book[file_name] = book_words(os.path.join(CLEAN_FOLDER, file_name))

# {word: number of books where it appears}:
books_per_word = Counter()
for words in words_by_book.values():
    books_per_word.update(set(words))
print(len(words_by_book), "books,", len(books_per_word), "different words")

151 books, 321099 different words


In [3]:
# Celda 3
# A word "does not exist" if it is neither in the Spanish dictionary nor in 10 books or more of the corpus:
# "eon" (for "con") does not exist; "dixo" and "Dios" do. Each different word is looked up once. A few minutes:
dictionary = Dictionary.from_files(DICTIONARY)
exists = {}
for word in books_per_word:
    exists[word] = books_per_word[word] >= MIN_BOOKS or dictionary.lookup(word)

# Share of words that do not exist, per book: the higher, the worse the OCR:
rows = []
for file_name, words in words_by_book.items():
    missing = sum(1 for word in words if not exists[word])
    rows.append({"book": file_name[:-4], "words": len(words), "unknown_pct": round(100 * missing / max(len(words), 1), 2)})
quality = pd.DataFrame(rows).sort_values("unknown_pct", ascending=False).reset_index(drop=True)
print(quality["unknown_pct"].describe().round(2))
quality.head(15)

count    151.00
mean       2.29
std        1.34
min        0.24
25%        1.38
50%        1.94
75%        2.94
max        8.23
Name: unknown_pct, dtype: float64


,book,words,unknown_pct
0,samaniego-prologo-galeato,51442,8.23
1,segneri-el-cristiano-instruido-en-su-ley-tomo2,140864,7.58
2,torrubia-practica-de-los-ejercicios-espiritual...,181540,6.36
3,segneri-el-cristiano-instruido-en-su-ley-tomo4,100433,5.64
4,agreda-cartas-a-felipe-iv-tomo1,136020,4.92
5,segneri-el-cristiano-instruido-en-su-ley-tomo1,227902,4.89
6,isla-cartas-de-juan-de-la-encina,21200,4.58
7,loyola-ejercicios-espirituales,24688,4.50
8,agreda-cartas-a-felipe-iv-tomo2,251797,4.48
9,carvajal-las-cortes-de-la-muerte,38997,4.48


In [4]:
# Celda 4
# Pages of a clean book: {page number: text}. Each page starts with <pb n="N"/>, N being the page of the PDF:
def read_pages(path):
    with open(path, encoding="utf-8") as f:
        pieces = re.split(r'<pb n="(\d+)"/>\n?', f.read())
    pages = {}
    for k in range(1, len(pieces), 2):
        pages[int(pieces[k])] = pieces[k + 1].strip("\n")
    return pages

# One random page with MIN_WORDS words or more from each of the worst books. The draw was made once,
# and its pages were then transcribed by hand (ocr-piloto/referencia-cer), together with other pages chosen later.
# Since any change in the cleaning changes the word counts and so the draw, the sample is now every transcribed page
# ("<book>_p<page>.txt") whose book is still in the corpus. To draw again, set DRAW = True:
DRAW = False
FIXED_PAGES = []
for file_name in sorted(os.listdir("ocr-piloto/referencia-cer")):
    book, number = re.match(r"(.+)_p(\d+)\.txt$", file_name).groups()
    if os.path.exists(os.path.join(CLEAN_FOLDER, book + ".txt")):
        FIXED_PAGES.append((book, int(number)))
    else:
        print("out of the corpus, not measured:", file_name)

sample = []
if DRAW:
    random.seed(SEED)
    for book in quality["book"].head(WORST_BOOKS):
        pages = read_pages(os.path.join(CLEAN_FOLDER, book + ".txt"))
        candidates = [number for number, text in pages.items() if len(WORD.findall(text)) >= MIN_WORDS]
        number = random.choice(candidates)
        sample.append({"book": book, "page": number, "text": pages[number]})
else:
    # A page whose text 5-limpieza now removes is not measured: fewer than half the words of its reference transcription are left
    for book, number in FIXED_PAGES:
        pages = read_pages(os.path.join(CLEAN_FOLDER, book + ".txt"))
        text = pages.get(number, "")
        with open(os.path.join("ocr-piloto/referencia-cer", f"{book}_p{number}.txt"), encoding="utf-8") as f:
            reference_words = len(WORD.findall(f.read()))
        if len(WORD.findall(re.sub(r"<[^>]+>", " ", text))) < reference_words / 2:
            print("text removed by the cleaning, not measured:", book, number)
            continue
        sample.append({"book": book, "page": number, "text": text})
pd.DataFrame(sample)[["book", "page"]]

,book,page
0,battista-da-crema-tratado-de-la-victoria-de-si...,25
1,cabrera-consideraciones-sobre-los-evangelios-d...,216
2,guevara-epistolas-familiares,309
3,juan-de-la-cruz-cartas,15
4,juan-de-la-cruz-llama-de-amor-viva-segunda-red...,7
5,luis-de-la-palma-historia-de-la-sagrada-pasion,8
6,luis-de-leon-de-los-nombres-de-cristo,317
7,luis-de-leon-poesias,22
8,pinamonti-breves-meditaciones-sobre-los-novisimos,13
9,ribera-vida-de-santa-teresa-de-jesus,377


In [5]:
# Celda 5
# For each page: the image of the PDF page (to read it by eye) and its clean text.
# "agreda-...-1885" page 12 -> "agreda-...-1885_p12.png" and "agreda-...-1885_p12_limpio.txt":
os.makedirs(OUT_FOLDER, exist_ok=True)
for page in sample:
    name = os.path.join(OUT_FOLDER, page["book"] + "_p" + str(page["page"]))
    with pymupdf.open(os.path.join(PDF_FOLDER, page["book"] + ".pdf")) as pdf:
        pdf[page["page"] - 1].get_pixmap(dpi=150).save(name + ".png")
    with open(name + "_limpio.txt", "w", encoding="utf-8") as f:
        f.write(page["text"])
print(len(sample), "pages saved in", OUT_FOLDER)

17 pages saved in ocr-piloto/paginas-baja-calidad


In [6]:
# Celda 6
# The same modernization and post-correction as 6-modernizacion, on these pages only, with the corpus vocabulary that
# 6-modernizacion saved (frequent OCR errors, drop caps, Latin words included): run 6-modernizacion first.
VOCABULARY_FILE = os.path.join(tempfile.gettempdir(), "vocabulario-corpus.pkl")
with open(VOCABULARY_FILE, "rb") as f:
    vocabulary = pickle.load(f)
reglas.iniciar(VOCABULARY_FILE)  # Loads the vocabulary for modernizar_libro, here in this same process

# Steps 1-4 and 6 (as Celda 3 of 6-modernizacion), "..._limpio.txt" -> "..._modernizado.txt", with the register
# of changes of each page in "..._cambios.csv":
for page in sample:
    name = os.path.join(OUT_FOLDER, page["book"] + "_p" + str(page["page"]))
    print(reglas.modernizar_libro((name + "_limpio.txt", name + "_modernizado.txt", name + "_cambios.csv")))

('battista-da-crema-tratado-de-la-victoria-de-si-mismo_p25_modernizado.txt', {'sc': 1})


('cabrera-consideraciones-sobre-los-evangelios-de-la-cuaresma_p216_modernizado.txt', {'contracción': 1})
('guevara-epistolas-familiares_p309_modernizado.txt', {'contracción': 1})


('juan-de-la-cruz-cartas_p15_modernizado.txt', {})
('juan-de-la-cruz-llama-de-amor-viva-segunda-redaccion_p7_modernizado.txt', {'tilde': 1})
('luis-de-la-palma-historia-de-la-sagrada-pasion_p8_modernizado.txt', {})


('luis-de-leon-de-los-nombres-de-cristo_p317_modernizado.txt', {})
('luis-de-leon-poesias_p22_modernizado.txt', {'contracción': 3, 'tilde': 4})
('pinamonti-breves-meditaciones-sobre-los-novisimos_p13_modernizado.txt', {})


('ribera-vida-de-santa-teresa-de-jesus_p377_modernizado.txt', {})
('rosignoli-verdades-eternas_p332_modernizado.txt', {})


('samaniego-prologo-galeato_p77_modernizado.txt', {'tilde': 1})


('samaniego-relacion-de-la-vida-de-sor-maria-de-jesus_p11_modernizado.txt', {})


('segneri-el-cristiano-instruido-en-su-ley-tomo2_p334_modernizado.txt', {})


('teresa-de-jesus-exclamaciones-del-alma-a-dios_p24_modernizado.txt', {'contracción': 1})


('teresa-de-jesus-modo-de-visitar-los-conventos_p6_modernizado.txt', {})


('torrubia-practica-de-los-ejercicios-espirituales-de-san-ignacio_p656_modernizado.txt', {})


In [7]:
# Celda 7
# Step 5 (as Celda 4 of 6-modernizacion): OCR errors one typical confusion away from a common word, only if the
# corrected word appears in the corpus next to the same neighbor:
needed = set()
for page in sample:
    name = os.path.join(OUT_FOLDER, page["book"] + "_p" + str(page["page"]))
    with open(name + "_modernizado.txt", encoding="utf-8") as f:
        for line in f:
            reglas.pares_necesarios(line.rstrip("\n"), vocabulary, needed)
pairs = reglas.contar_pares(CLEAN_FOLDER, needed)

PROTECTED = reglas.patron_protegido(vocabulary)
stats = Counter()
for page in sample:
    path = os.path.join(OUT_FOLDER, page["book"] + "_p" + str(page["page"]) + "_modernizado.txt")
    with open(path, encoding="utf-8") as f:
        lines = f.read().split("\n")
    for k in range(len(lines)):
        lines[k] = reglas.procesar_linea(lines[k], PROTECTED, lambda s: reglas.corregir_ocr(s, vocabulary, pairs, stats))
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
print(stats["corregida"], "OCR errors corrected")

14 OCR errors corrected


In [8]:
# Celda 8
# Share of words that do not exist on each page, before (clean) and after (modernized):
def unknown_pct(text):
    words = [word.lower() for word in WORD.findall(re.sub(r"<[^>]+>", " ", text))]
    missing = 0
    for word in words:
        if word not in exists:
            exists[word] = dictionary.lookup(word)  # A form the modernization created, not in the clean texts
        if not exists[word]:
            missing = missing + 1
    return round(100 * missing / max(len(words), 1), 2)

for page in sample:
    name = os.path.join(OUT_FOLDER, page["book"] + "_p" + str(page["page"]))
    with open(name + "_modernizado.txt", encoding="utf-8") as f:
        page["unknown_pct_modern"] = unknown_pct(f.read())
    page["unknown_pct_clean"] = unknown_pct(page["text"])
pd.DataFrame(sample)[["book", "page", "unknown_pct_clean", "unknown_pct_modern"]]

,book,page,unknown_pct_clean,unknown_pct_modern
0,battista-da-crema-tratado-de-la-victoria-de-si...,25,0.40,0.81
1,cabrera-consideraciones-sobre-los-evangelios-d...,216,4.31,4.10
2,guevara-epistolas-familiares,309,3.27,3.26
3,juan-de-la-cruz-cartas,15,0.96,0.48
4,juan-de-la-cruz-llama-de-amor-viva-segunda-red...,7,0.63,0.31
5,luis-de-la-palma-historia-de-la-sagrada-pasion,8,0.00,0.00
6,luis-de-leon-de-los-nombres-de-cristo,317,0.00,0.00
7,luis-de-leon-poesias,22,6.02,5.58
8,pinamonti-breves-meditaciones-sobre-los-novisimos,13,1.20,1.20
9,ribera-vida-de-santa-teresa-de-jesus,377,0.25,0.25


In [9]:
# Celda 9
# CER (character error rate) against a reference transcription of each page, made by reading the page image:
# ocr-piloto/referencia-cer/<book>_p<page>.txt. Only the body text: no running heads, page numbers, footnotes or
# margin notes, as 5-limpieza removes them; spelling as printed. Tags and extra spaces are left out before comparing.
# "no_accents" also leaves out accents ("Vírgen" = "Virgen"), so it does not count the modernization's accents.
# About a minute (the edit distance is computed by hand, character by character):
import unicodedata
REFERENCE_FOLDER = "ocr-piloto/referencia-cer"

def normalize_for_cer(text, keep_accents):
    text = re.sub(r"<[^>]+>", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not keep_accents:
        letters = unicodedata.normalize("NFD", text)
        text = "".join(c for c in letters if unicodedata.category(c) != "Mn")
    return text

# Fewest insertions, deletions and substitutions that turn a into b: "casa", "cosas" -> 2:
def edit_distance(a, b):
    previous = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        current = [i]
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + cost))
        previous = current
    return previous[-1]

def cer(reference, text, keep_accents):
    reference = normalize_for_cer(reference, keep_accents)
    text = normalize_for_cer(text, keep_accents)
    return round(100 * edit_distance(reference, text) / len(reference), 2)

rows = []
for page in sample:
    name = page["book"] + "_p" + str(page["page"])
    with open(os.path.join(REFERENCE_FOLDER, name + ".txt"), encoding="utf-8") as f:
        reference = f.read()
    with open(os.path.join(OUT_FOLDER, name + "_modernizado.txt"), encoding="utf-8") as f:
        modern = f.read()
    rows.append({"book": page["book"][:40], "page": page["page"],
                 "cer_clean_no_accents": cer(reference, page["text"], False),
                 "cer_modern_no_accents": cer(reference, modern, False),
                 "cer_clean": cer(reference, page["text"], True),
                 "cer_modern": cer(reference, modern, True)})
cer_table = pd.DataFrame(rows)
print(cer_table.mean(numeric_only=True).drop("page").round(2))
cer_table

cer_clean_no_accents     0.74
cer_modern_no_accents    0.68
cer_clean                0.87
cer_modern               1.33
dtype: float64


,book,page,cer_clean_no_accents,cer_modern_no_accents,cer_clean,cer_modern
0,battista-da-crema-tratado-de-la-victoria,25,0.45,0.23,0.45,1.43
1,cabrera-consideraciones-sobre-los-evange,216,0.66,0.66,0.79,1.12
2,guevara-epistolas-familiares,309,0.14,0.23,0.28,1.16
3,juan-de-la-cruz-cartas,15,0.80,0.80,0.80,1.51
4,juan-de-la-cruz-llama-de-amor-viva-segun,7,0.00,0.00,0.46,0.12
5,luis-de-la-palma-historia-de-la-sagrada-,8,0.47,0.54,0.47,0.61
6,luis-de-leon-de-los-nombres-de-cristo,317,0.00,0.00,0.10,0.00
7,luis-de-leon-poesias,22,0.88,0.88,0.95,1.58
8,pinamonti-breves-meditaciones-sobre-los-,13,1.26,1.05,1.57,1.89
9,ribera-vida-de-santa-teresa-de-jesus,377,0.34,0.34,0.34,1.06
